# tensor-zeros-init — ex3: zeros_like — mirror an input's shape and dtype

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-zeros-init`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five allocation patterns that ramp from `torch.zeros(n)` → multi-axis shape → `zeros_like` → dtype-long index buffer → allocate-then-scatter for the canonical Ray Tracing per-ray output-buffer pattern. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Core array literacy` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `tensor-zeros-init`**, which bridges to the bank subtopic `Numpy: Core array literacy` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-zeros-init"
DD_SUBTOPIC = "Numpy: Core array literacy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Tensor allocation — quick refresher

**The four shapes of `zeros`:**
- `t.zeros(n)` — 1-D, shape `(n,)`, default `float32`.
- `t.zeros(b, h, w)` — multi-axis positional args.
- `t.zeros_like(x)` — mirror `x.shape` + `x.dtype` + `x.device`.
- `t.zeros(n, dtype=t.long)` — override dtype for index buffers.

**The accumulator pattern.** Allocate the right-shaped zero buffer first; scatter per-element results into it via indexed assignment. Cleaner and faster than `append`-and-stack.

### Exercise 3 — zeros_like — mirror an input's shape and dtype

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `torch.zeros_like` to allocate a fresh zero buffer matching the input's shape AND dtype.
> Keywords: zeros-like, shape-mirror, dtype-mirror
> ```

**KCs targeted:** `zeros-like-mirrors-input`

Implement `ex3_zeros_like(x)` to return a fresh zero tensor with the same shape, dtype, and device as `x`.

Use `torch.zeros_like(x)`. Critically: the result must NOT be a view or alias of `x` — writing to the output must not change `x`.

In [ ]:
def ex3_zeros_like(x: Tensor) -> Tensor:
    """Return a fresh zero buffer mirroring x.shape and x.dtype."""
    raise NotImplementedError()


def _test_ex3():
    x_int = t.tensor([[1, 2, 3], [4, 5, 6]], dtype=t.int64)
    out = ex3_zeros_like(x_int)
    assert out.shape == x_int.shape, f'shape mismatch: {out.shape} vs {x_int.shape}'
    assert out.dtype == t.int64, f'dtype must be mirrored: got {out.dtype}'
    assert t.all(out == 0), 'must be all zeros'
    # Aliasing check — writing to out must not mutate x.
    out[0, 0] = 99
    assert x_int[0, 0].item() == 1, 'zeros_like must be a FRESH tensor, not a view'

    x_float = t.randn(4, 5)
    out_f = ex3_zeros_like(x_float)
    assert out_f.shape == x_float.shape
    assert out_f.dtype == t.float32, 'float32 input → float32 output'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_zeros_like(x: Tensor) -> Tensor:
    return t.zeros_like(x)
```

**Why prefer `zeros_like(x)` over `zeros(x.shape)`?**
- `zeros(x.shape)` only copies the shape — the dtype reverts to float32 and the device reverts to CPU. If `x` is a `int64` GPU tensor, your buffer ends up float32 on CPU — silent breakage the moment you try to use it as indices or do an op against `x`.
- `zeros_like(x)` mirrors shape + dtype + device. Always the right call when allocating a per-input accumulator.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()